In [0]:
from msal import ConfidentialClientApplication

scope = "fulfilment-fabric"

app = ConfidentialClientApplication(
    client_id=dbutils.secrets.get(scope, "client-id").strip(),
    authority=(
        "https://login.microsoftonline.com/"
        + dbutils.secrets.get(scope, "tenant-id").strip()
    ),
    client_credential=dbutils.secrets.get(scope, "client-secret").strip()
)

result = app.acquire_token_for_client(
    scopes=["https://onelake.fabric.microsoft.com/.default"]
)

if "access_token" in result:
    print("Authentication successful!")
else:
    print("Authentication failed:", result.get("error"))
    print("Error codes:", result.get("error_codes", []))

In [0]:
%pip install msal deltalake pyarrow

In [0]:
import pyarrow as pa
from deltalake import write_deltalake

tables = [
    "gold_daily_orders",
    "gold_hourly_orders",
    "gold_orders_by_zone",
    "gold_daily_weather",
    "gold_dim_zone"
]

lakehouse_path = (
    "abfss://2746cbce-576c-4826-834d-cadb47c14b7c"
    "@onelake.dfs.fabric.microsoft.com/"
    "1ee02e29-278c-474c-993d-d6a9c64f60a0/"
    "Tables/dbo"
)

# Check and prepare all five small sample tables before writing
prepared = {}

for table_name in tables:
    source = spark.table(
        f"dp_750.default.{table_name}"
    ).toPandas()

    if source.empty:
        raise ValueError(f"{table_name} is empty. Publishing stopped.")

    prepared[table_name] = pa.Table.from_pandas(
        source,
        preserve_index=False
    )

# Publish each table
for table_name, data in prepared.items():
    result = app.acquire_token_for_client(
        scopes=["https://onelake.fabric.microsoft.com/.default"]
    )

    if "access_token" not in result:
        raise RuntimeError("Authentication failed. Publishing stopped.")

    write_deltalake(
        f"{lakehouse_path}/{table_name}",
        data,
        mode="overwrite",
        schema_mode="overwrite",
        storage_options={
            "bearer_token": result["access_token"],
            "use_fabric_endpoint": "true"
        }
    )

    print(f"Published {table_name}: {data.num_rows} rows")

print("All five tables published successfully.")